# Pre-entrega 6 — Orquestador multiagente

## Objetivo
Demostrar delegacion real a un investigador y un analista, con herramientas funcionales y un supervisor que exige validacion antes de finalizar.

El corpus de cuatro documentos es original y educativo. La ejecucion usa DeepSeek Flash real, embeddings ONNX existentes, Chroma y Phoenix. No guarda claves ni descarga modelos.

## Configuracion
Ejecutar desde la raiz con requirements.lock.txt ya instalado. La credencial autorizada llega solamente al entorno del proceso; nunca se muestra ni se incorpora al notebook. Los embeddings ya deben estar en cache. No hace falta que FastAPI este activa para este ejemplo.

In [1]:
from app.llm import ModelGateway
from app.rag import HybridKnowledgeBase
from app.graph import build_graph
from langchain_core.messages import HumanMessage

gateway = ModelGateway()
await gateway.validate_environment()
knowledge = HybridKnowledgeBase(gateway)
await knowledge.setup()
graph = build_graph(gateway, knowledge)
print('Documentos fragmentados:', len(knowledge.chunks))

Documentos fragmentados: 4


## Ejecucion y delegacion
La pregunta requiere recuperar la politica de persistencia y analizar como se retoma una ejecucion. Se guardan decisiones operativas y aportes, no razonamiento privado del modelo.

In [2]:
query = 'Como se conserva un trabajo cuando reinicia FastAPI y como lo valida el supervisor?'
result = await graph.ainvoke({'query': query, 'messages': [HumanMessage(content=query)], 'contributions': [], 'usages': [], 'routing_log': [], 'steps': 0})
for decision in result['routing_log']:
    print(decision['step'], decision['selected'], '-', decision['reason'])
print('\nAportes y herramientas')
for contribution in result['contributions']:
    print(contribution['agent'], '->', contribution['tool'])

1 researcher - No hay evidencia ni análisis previos; se requiere investigar primero cómo se conserva un trabajo al reiniciar FastAPI y cómo lo valida el supervisor.
2 analyst - La evidencia de investigación ya está disponible; falta el análisis para interpretar la conservación del trabajo y la validación del supervisor antes de pasar a validación.
3 validate - Investigación y análisis ya están disponibles y no hay errores; corresponde validar los artefactos antes de decidir una ronda de refinamiento.

Aportes y herramientas
researcher -> search_knowledge_base
analyst -> validate_research_schema


## Resultado validado
Las referencias deben existir en los fragmentos recuperados. Una respuesta afirmativa sin referencias no puede superar el validador.

In [3]:
from pprint import pprint
pprint(result['final'])
assert result['final']['validation_passed']
assert set(result['final']['contributors']) == {'researcher', 'analyst'}
assert all(r['id'] in {d['id'] for d in result['evidence']} for r in result['final']['references'])

{'answer': 'Según la evidencia disponible, la conservación del trabajo ante '
           'reinicios se apoya en los checkpoints de LangGraph guardados en '
           'Redis con thread_id igual a job_id. Redis debe usar appendonly yes '
           'para sobrevivir a su propio reinicio, y reiniciar solamente '
           'FastAPI no borra los resultados ni las conversaciones. Además, al '
           'reiniciar un worker aislado, este recupera los trabajos RUNNING '
           'pendientes desde el último checkpoint. Sobre la validación del '
           'supervisor, el manual de arquitectura indica que el supervisor no '
           'puede declarar completa una respuesta sin pasar por la validación '
           'final, y que el analista valida la estructura de la respuesta. Sin '
           'embargo, los documentos no detallan el procedimiento exacto con el '
           'que el supervisor valida la conservación del trabajo tras el '
           'reinicio de FastAPI.',
 'answerable': False,


## Tokens y costo
Los tokens LLM son informados por DeepSeek, incluyendo intentos invalidos antes de un reintento exitoso. Los embeddings ONNX no tienen un contador comparable; no se inventa. El costo usa tarifas Flash OFF-PEAK verificadas el 26 de septiembre de 2026; es una estimacion y no una factura.

In [4]:
from app.observability import usage_cost
print('Tokens entrada:', sum(u.get('input_tokens', 0) for u in result['usages']))
print('Tokens salida:', sum(u.get('output_tokens', 0) for u in result['usages']))
print('Costo de esta ejecucion USD:', round(sum(usage_cost(u) for u in result['usages']), 6))
await gateway.close()

Tokens entrada: 4485
Tokens salida: 663
Costo de esta ejecucion USD: 0.000826


## Conclusiones
El supervisor delega segun los artefactos disponibles. El investigador consulta la base hibrida y el analista verifica el esquema y las fuentes. El nodo final valida el contrato y produce una respuesta o una limitacion. Los ciclos estan acotados a una ronda de refinamiento y ocho decisiones.

Las pruebas adicionales de concurrencia, interrupcion y reinicio estan en tests/test_system.py y evidence/benchmark.json.